# FinGuard ID — Inference (Revisi 4: Dual Regression)

Notebook ini menguji kedua model dengan data yang **tidak dipakai saat training**, dengan cara yang sama seperti aplikasi Streamlit dan DAG Airflow memakainya:
1. Input hanya **11 kolom mentah**; UMR diisi dari provinsi (`umr_reference.json`).
2. **Model 1** (`model_kelayakan.pkl`) → prediksi DPD → kolektibilitas & status kelayakan (batas dari CV di `model_metadata.json`), 4 flag OJK, 3 faktor SHAP, rekomendasi.
3. **Model 2** (`model_restrukturisasi.pkl`) → simulasi tenor untuk prediksi DPD 31–90 → tabel **Before vs After**.

**5 kasus uji.** Kasus 1, 2, 3, dan 5 dipilih otomatis dari **data test** berdasarkan kriteria; kasus 4 dibuat manual. Kriteria hanya menentukan *siapa* yang diuji. **Hasilnya dilaporkan apa adanya, tidak diandaikan.**

## 1. Load model, metadata & referensi UMR

In [1]:
import json
import os
import warnings
os.environ['PYTHONWARNINGS'] = 'ignore'
warnings.filterwarnings('ignore', message=r'X does not have valid feature names.*LGBMRegressor', category=UserWarning)
warnings.filterwarnings('ignore', category=UserWarning)
import joblib
import numpy as np
import pandas as pd
import shap
import sklearn
from features import (DSR_CAP, FLAG_COLS, FLAG_INFO, RAW_INPUT, TARGET, SIM_DISCLAIMER, SIM_NO_OPTION, add_features,
                      before_after_table, dpd_to_collectibility, dpd_to_eligibility, max_principal_ojk, predict_dpd,
                      recommend, simulate_restructuring, top_factors_from_shap)
pd.set_option('display.width', 220); pd.set_option('display.max_columns', 30)

m1 = joblib.load('model/model_kelayakan.pkl')
m2 = joblib.load('model/model_restrukturisasi.pkl')
with open('model/model_metadata.json') as f:
    meta = json.load(f)
with open('model/umr_reference.json') as f:
    umr = json.load(f)
THR = meta['thresholds']
print('Model 1:', meta['model_1']['algoritma'], '| test', meta['model_1']['test'])
print('Model 2:', meta['model_2']['algoritma'], '| test', meta['model_2']['test'])
print('Batas pemetaan:', THR)
print('scikit-learn: training', meta['versions']['scikit-learn'], '| sekarang', sklearn.__version__)
assert meta['versions']['scikit-learn'] == sklearn.__version__, 'Versi scikit-learn berbeda dengan saat training!'
explainer = shap.TreeExplainer(m1['reg'].regressor_)

Model 1: LightGBM | test {'MAE': 8.471, 'RMSE': 12.286, 'R2': 0.957}
Model 2: LightGBM | test {'MAE': 10.68, 'RMSE': 13.953, 'R2': 0.94}
Batas pemetaan: {'lancar': 8.0, 'dpk': 30, 'kurang_lancar': 60, 'diragukan': 90}
scikit-learn: training 1.6.1 | sekarang 1.6.1


## 2. Fungsi bantu
* `cek_rentang`: peringatan jika input di luar rentang data train (prediksinya ekstrapolasi).
* `prediksi`: UMR dari provinsi → Model 1 → DPD, kolektibilitas, status, 4 flag, SHAP, rekomendasi.
* `simulasi_restrukturisasi`: memanggil `features.simulate_restructuring` dan `before_after_table`, fungsi yang sama dengan aplikasi dan DAG.

In [2]:
def cek_rentang(row):
    return [f"{c} = {float(row[c].iloc[0]):,.2f} di luar rentang train ({r['min']:,.2f} – {r['max']:,.2f})"
            for c, r in meta['input_range'].items() if not r['min'] <= float(row[c].iloc[0]) <= r['max']]


def prediksi(data):
    data = dict(data)
    if 'province' in data:
        data['regional_minimum_wage_umr_idr'] = umr[data.pop('province')]
    row = pd.DataFrame([data])[RAW_INPUT]
    f = add_features(row).iloc[0]
    dpd = float(predict_dpd(m1, row)[0])
    kelas, status = dpd_to_collectibility([dpd], THR)[0], dpd_to_eligibility([dpd], THR)[0]
    Xt = m1[:-1].transform(row); Xt = Xt.toarray() if hasattr(Xt, 'toarray') else Xt
    fac = top_factors_from_shap(explainer.shap_values(Xt)[0], meta['feature_names_out'], row, top=3)
    print(f"Prediksi DPD: {dpd:.1f} hari | Kolektibilitas: {kelas} | Status: {status}")
    print(f"Cicilan Rp{f['monthly_installment_estimate_idr']:,.0f} | DSR {f['debt_service_ratio']:.1%} | aplikasi {int(f['num_active_lending_apps'])} | tenor {int(f['loan_tenor_months'])} bln")
    print('3 faktor yang paling menaikkan prediksi DPD (SHAP):')
    for x in fac:
        print(f"   + {x['label']:<33} = {x['nilai']:<34} {x['shap_hari']:+.1f} hari")
    print('Kepatuhan OJK:', {FLAG_INFO[c][0][:40]: ('TIDAK SESUAI' if f[c] else 'sesuai') for c in FLAG_COLS})
    print('Rekomendasi:', recommend(kelas, f['flag_dsr_above_30']))
    for w in cek_rentang(row):
        print('⚠️  PERINGATAN:', w)
    return row, dpd


def simulasi_restrukturisasi(row, dpd):
    if not 30 < dpd <= 90:
        print(f'Prediksi DPD {dpd:.1f} hari di luar 31–90 → simulasi restrukturisasi tidak dijalankan.')
        return None
    s = simulate_restructuring(row, m2, THR)
    if len(s['options']):
        display(s['options'])
    if s['best'] is not None:
        print(f"✅ Opsi disarankan: tenor {int(s['best']['tenor_bulan'])} bulan, DSR {s['best']['dsr']:.1%}, "
              f"prediksi DPD after {s['best']['pred_dpd_after']:.1f} hari ({s['best']['kolektibilitas_after']})")
    else:
        h = s['haircut']
        print('❌', SIM_NO_OPTION)
        print(f"   Pokok maksimum OJK (DSR 30% @ tenor 24): Rp{h['pokok_maks_ojk_idr']:,.0f} | pokok sekarang Rp{h['pokok_sekarang_idr']:,.0f} | "
              f"kelebihan Rp{h['kelebihan_pokok_idr']:,.0f} ({h['potongan_pct']:.1f}%) | skenario tenor 24 + pokok disesuaikan → "
              f"DPD {h['pred_dpd_after']:.1f} hari ({h['kolektibilitas_after']})")
    display(before_after_table(row, m1, s, THR))
    print('Disclaimer:', SIM_DISCLAIMER)
    return s

## 3. Memilih kasus uji dari data test
Data test = baris yang `borrower_id`-nya ada di `model/test_ids.csv`, sehingga tidak pernah dilihat model saat training. Kolom `days_past_due` aktual hanya dipakai untuk **membandingkan** hasil, bukan sebagai input.

In [3]:
raw = pd.read_csv('pipeline/data/finguard_raw.csv')
test = raw[raw['borrower_id'].isin(pd.read_csv('model/test_ids.csv')['borrower_id'])].reset_index(drop=True)
tf = add_features(test)
test['pred'] = predict_dpd(m1, test)

# simulasi cepat untuk kandidat 31–90 (dipakai memilih kasus 2, 3, 5)
mid = test[(test['pred'] > 30) & (test['pred'] <= 90)].copy()
sims = {i: simulate_restructuring(test.loc[[i]], m2, THR) for i in mid.index}
mid['punya_opsi'] = [not sims[i]['need_other'] for i in mid.index]
mid['kelebihan_pokok'] = [sims[i]['haircut']['kelebihan_pokok_idr'] if sims[i]['haircut'] else 0 for i in mid.index]
mid['dsr'] = tf.loc[mid.index, 'debt_service_ratio']

pilih = {
    1: tf[(tf['debt_service_ratio'] <= DSR_CAP) & (tf['num_active_lending_apps'] <= 3)].index[0],
    2: mid[(mid['pred'] <= 60) & mid['punya_opsi']].index[0],
    3: mid[(mid['pred'] > 60) & mid['punya_opsi']].index[0],
    5: mid[~mid['punya_opsi'] & (mid['kelebihan_pokok'] > 0)].sort_values('dsr', ascending=False).index[0],
}
kriteria = {1: 'DSR ≤ 30% dan ≤ 3 aplikasi (baris pertama)', 2: 'prediksi DPD 31–60 (baris pertama yang punya opsi tenor)',
            3: 'prediksi DPD 61–90 (baris pertama yang punya opsi tenor)', 5: 'tidak punya opsi tenor & DSR masih > 30% di tenor 24 (butuh keringanan pokok), DSR tertinggi'}
display(pd.DataFrame({'kasus': list(pilih), 'kriteria': [kriteria[k] for k in pilih],
                      'borrower_id': [test.loc[i, 'borrower_id'] for i in pilih.values()],
                      'DPD aktual': [test.loc[i, TARGET] for i in pilih.values()]}))
print(f"Kandidat 31–90: {len(mid)} | tanpa opsi tenor: {(~mid['punya_opsi']).sum()} | di antaranya butuh keringanan pokok: {(~mid['punya_opsi'] & (mid['kelebihan_pokok'] > 0)).sum()}")
def data_kasus(i):
    return test.loc[i, RAW_INPUT].to_dict()

,kasus,kriteria,borrower_id,DPD aktual
0,1,DSR ≤ 30% dan ≤ 3 aplikasi (baris pertama),ID_DEB_5133A74CB9,0
1,2,prediksi DPD 31–60 (baris pertama yang punya o...,ID_DEB_CB8E5E4C69,54
2,3,prediksi DPD 61–90 (baris pertama yang punya o...,ID_DEB_4F9B14BC97,81
3,5,tidak punya opsi tenor & DSR masih > 30% di te...,ID_DEB_EB8AE760DA,69


Kandidat 31–90: 409 | tanpa opsi tenor: 87 | di antaranya butuh keringanan pokok: 7


### Kasus 1 — Profil sehat (DSR ≤ 30%, ≤ 3 aplikasi)

In [4]:
row1, d1 = prediksi(data_kasus(pilih[1])); print('DPD aktual:', test.loc[pilih[1], TARGET])
_ = simulasi_restrukturisasi(row1, d1)

Prediksi DPD: 0.0 hari | Kolektibilitas: Lancar | Status: Layak
Cicilan Rp468,977 | DSR 11.6% | aplikasi 2 | tenor 6 bln
3 faktor yang paling menaikkan prediksi DPD (SHAP):
   + Tenor (bulan)                     = 6                                  +2.1 hari
   + Penghasilan bulanan               = Rp4,060,000                        +0.9 hari
   + Limit kredit                      = Rp8,640,000                        +0.6 hari
Kepatuhan OJK: {'Bunga harian di atas batas OJK (0,3%/har': 'sesuai', 'Penghasilan di bawah Rp3 juta/bulan': 'sesuai', 'Indikator risiko multi-platform: > 3 apl': 'sesuai', 'Cicilan bulanan > 30% penghasilan (DSR)': 'sesuai'}
Rekomendasi: Pemantauan rutin
DPD aktual: 0
Prediksi DPD 0.0 hari di luar 31–90 → simulasi restrukturisasi tidak dijalankan.


**Hasil:** prediksi **0,0 hari → Lancar, Layak** (DPD aktual 0 ✅). DSR 11,6%, 2 aplikasi, patuh keempat aturan OJK. Faktor penaik DPD sangat kecil (+2,1 hari dari tenor 6 bulan). Simulasi restrukturisasi tidak dijalankan karena prediksi di luar 31–90 hari, dan rekomendasinya cukup pemantauan rutin.

### Kasus 2 — Prediksi DPD 31–60 (Kurang Lancar) + simulasi

In [5]:
row2, d2 = prediksi(data_kasus(pilih[2])); print('DPD aktual:', test.loc[pilih[2], TARGET])
s2 = simulasi_restrukturisasi(row2, d2)

Prediksi DPD: 47.4 hari | Kolektibilitas: Kurang Lancar | Status: Dalam Pengawasan
Cicilan Rp3,616,800 | DSR 108.6% | aplikasi 4 | tenor 1 bln
3 faktor yang paling menaikkan prediksi DPD (SHAP):
   + Rasio cicilan/penghasilan (DSR)   = 109%                               +16.3 hari
   + Limit kredit                      = Rp6,080,000                        +5.4 hari
   + > 3 aplikasi aktif                = ya                                 +2.9 hari
Kepatuhan OJK: {'Bunga harian di atas batas OJK (0,3%/har': 'TIDAK SESUAI', 'Penghasilan di bawah Rp3 juta/bulan': 'sesuai', 'Indikator risiko multi-platform: > 3 apl': 'TIDAK SESUAI', 'Cicilan bulanan > 30% penghasilan (DSR)': 'TIDAK SESUAI'}
Rekomendasi: Tawaran restrukturisasi (simulasi Model 2) | Catatan: DSR melebihi batas OJK 30%
DPD aktual: 54


,tenor_bulan,bunga_harian_pct,cicilan_bulanan_idr,dsr,dsr_sesuai_ojk,pred_dpd_after,kolektibilitas_after,memenuhi_syarat
0,3,0.3,1397000,0.4195,False,20.1,DPK,False
1,6,0.3,847000,0.2544,True,17.6,DPK,True
2,9,0.2,564667,0.1696,True,14.2,DPK,True
3,12,0.2,473000,0.1420,True,14.1,DPK,True
4,18,0.2,381333,0.1145,True,15.8,DPK,True
5,24,0.2,335500,0.1008,True,15.8,DPK,True


✅ Opsi disarankan: tenor 6 bulan, DSR 25.4%, prediksi DPD after 17.6 hari (DPK)


,Before,After (tenor disarankan)
Tenor (bulan),1,6
Bunga harian (%),0.32,0.3
Pokok (Rp),3300000,3300000
Cicilan bulanan (Rp),3616800,847000
DSR,108.6%,25.4%
Prediksi DPD (hari),47.4,17.6
Kolektibilitas,Kurang Lancar,DPK
Status,Dalam Pengawasan,Layak


Disclaimer: Simulasi menunjukkan perubahan prediksi model jika tenor atau pokok diubah. Model mempelajari pola dari data, bukan hubungan sebab-akibat, sehingga hasil simulasi bukan jaminan debitur akan membayar. Keputusan keringanan pokok tetap memerlukan persetujuan manajemen.


**Hasil:** prediksi **47,4 hari → Kurang Lancar, Dalam Pengawasan** (aktual 54, error 6,6 hari). Faktor terbesar adalah **DSR 109%**: pinjaman Rp3,3 juta harus lunas dalam **1 bulan**. Debitur melanggar 3 aturan (bunga 0,32% > 0,3%, > 3 aplikasi, DSR).

**Simulasi:** tenor 3 bulan belum cukup (DSR 42%). **Tenor 6 bulan** adalah opsi pertama yang memenuhi dua syarat: cicilan Rp847.000, **DSR 25,4%**, dan prediksi DPD after **17,6 hari (DPK)**. Bunga ikut turun ke batas OJK 0,3%. Status berubah dari Dalam Pengawasan menjadi **Layak**.

### Kasus 3 — Prediksi DPD 61–90 (Diragukan) + simulasi

In [6]:
row3, d3 = prediksi(data_kasus(pilih[3])); print('DPD aktual:', test.loc[pilih[3], TARGET])
s3 = simulasi_restrukturisasi(row3, d3)

Prediksi DPD: 78.9 hari | Kolektibilitas: Diragukan | Status: Dalam Pengawasan
Cicilan Rp5,606,480 | DSR 186.9% | aplikasi 2 | tenor 1 bln
3 faktor yang paling menaikkan prediksi DPD (SHAP):
   + Rasio cicilan/penghasilan (DSR)   = 187%                               +57.6 hari
   + Penghasilan bulanan               = Rp3,000,000                        +3.1 hari
   + Bunga harian (%)                  = 0.36%                              +0.6 hari
Kepatuhan OJK: {'Bunga harian di atas batas OJK (0,3%/har': 'TIDAK SESUAI', 'Penghasilan di bawah Rp3 juta/bulan': 'sesuai', 'Indikator risiko multi-platform: > 3 apl': 'sesuai', 'Cicilan bulanan > 30% penghasilan (DSR)': 'TIDAK SESUAI'}
Rekomendasi: Tawaran restrukturisasi (simulasi Model 2) + kontak negosiasi langsung | Catatan: DSR melebihi batas OJK 30%
DPD aktual: 81


,tenor_bulan,bunga_harian_pct,cicilan_bulanan_idr,dsr,dsr_sesuai_ojk,pred_dpd_after,kolektibilitas_after,memenuhi_syarat
0,3,0.3,2142067,0.7140,False,18.4,DPK,False
1,6,0.3,1298733,0.4329,False,14.5,DPK,False
2,9,0.2,865822,0.2886,True,12.4,DPK,True
3,12,0.2,725267,0.2418,True,11.8,DPK,True
4,18,0.2,584711,0.1949,True,12.5,DPK,True
5,24,0.2,514433,0.1715,True,11.4,DPK,True


✅ Opsi disarankan: tenor 9 bulan, DSR 28.9%, prediksi DPD after 12.4 hari (DPK)


,Before,After (tenor disarankan)
Tenor (bulan),1,9
Bunga harian (%),0.36,0.2
Pokok (Rp),5060000,5060000
Cicilan bulanan (Rp),5606480,865822
DSR,186.9%,28.9%
Prediksi DPD (hari),78.9,12.4
Kolektibilitas,Diragukan,DPK
Status,Dalam Pengawasan,Layak


Disclaimer: Simulasi menunjukkan perubahan prediksi model jika tenor atau pokok diubah. Model mempelajari pola dari data, bukan hubungan sebab-akibat, sehingga hasil simulasi bukan jaminan debitur akan membayar. Keputusan keringanan pokok tetap memerlukan persetujuan manajemen.


**Hasil:** prediksi **78,9 hari → Diragukan** (aktual 81, error 2,1 hari). DSR **187%** menyumbang +57,6 hari.

**Simulasi:** tenor 3 dan 6 bulan masih DSR > 30%. **Tenor 9 bulan** (bunga dipotong dari 0,36% ke batas 0,2%) menghasilkan cicilan Rp865.822, **DSR 28,9%**, dan prediksi DPD after **12,4 hari (DPK)**. Penurunan sebesar ini terjadi karena masalah utama debitur hanyalah tenor 1 bulan, sementara jumlah aplikasinya hanya 2.

### Kasus 4 — Input di luar rentang data train (usia 42, penghasilan Rp45 juta)

In [7]:
kasus4 = {'age': 42, 'province': 'DKI Jakarta', 'education_level': 'Master (S2)', 'employment_status': 'Formal Private Employee',
          'monthly_income_idr': 45_000_000, 'num_active_lending_apps': 2, 'total_loan_outstanding_idr': 15_000_000,
          'credit_limit_granted_idr': 40_000_000, 'loan_tenor_months': 6, 'daily_interest_rate_pct': 0.15,
          'primary_loan_purpose': 'Gadget & Electronics Paylater'}
row4, d4 = prediksi(kasus4)
_ = simulasi_restrukturisasi(row4, d4)

Prediksi DPD: 0.0 hari | Kolektibilitas: Lancar | Status: Layak
Cicilan Rp3,175,000 | DSR 7.1% | aplikasi 2 | tenor 6 bln
3 faktor yang paling menaikkan prediksi DPD (SHAP):
   + Total pokok utang                 = Rp15,000,000                       +3.1 hari
   + Tenor (bulan)                     = 6                                  +1.5 hari
   + Bunga harian (%)                  = 0.15%                              +0.7 hari
Kepatuhan OJK: {'Bunga harian di atas batas OJK (0,3%/har': 'sesuai', 'Penghasilan di bawah Rp3 juta/bulan': 'sesuai', 'Indikator risiko multi-platform: > 3 apl': 'sesuai', 'Cicilan bulanan > 30% penghasilan (DSR)': 'sesuai'}
Rekomendasi: Pemantauan rutin
⚠️  PERINGATAN: age = 42.00 di luar rentang train (18.00 – 35.00)
⚠️  PERINGATAN: monthly_income_idr = 45,000,000.00 di luar rentang train (1,500,000.00 – 28,000,000.00)
Prediksi DPD 0.0 hari di luar 31–90 → simulasi restrukturisasi tidak dijalankan.


**Hasil:** model tetap memberi prediksi (**0,0 hari, Lancar**) tetapi memunculkan **2 peringatan**: usia 42 (data train 18–35) dan penghasilan Rp45 juta (maks. train Rp28 juta). Prediksi ini **ekstrapolasi**, jadi analis perlu mengecek manual. Aplikasi menampilkan peringatan yang sama.

### Kasus 5 — DSR sangat tinggi, tidak punya opsi tenor → "perlu penanganan lain" + pokok maksimum OJK

In [8]:
row5, d5 = prediksi(data_kasus(pilih[5])); print('DPD aktual:', test.loc[pilih[5], TARGET])
s5 = simulasi_restrukturisasi(row5, d5)
print(f"\nCek rumus: max_principal_ojk(penghasilan, 24) = Rp{max_principal_ojk(row5['monthly_income_idr'].iloc[0], 24):,.0f}")

Prediksi DPD: 83.8 hari | Kolektibilitas: Diragukan | Status: Dalam Pengawasan
Cicilan Rp3,763,597 | DSR 87.1% | aplikasi 5 | tenor 6 bln
3 faktor yang paling menaikkan prediksi DPD (SHAP):
   + Rasio cicilan/penghasilan (DSR)   = 87%                                +10.2 hari
   + Tenor (bulan)                     = 6                                  +9.1 hari
   + Total pokok utang                 = Rp15,970,000                       +8.1 hari
Kepatuhan OJK: {'Bunga harian di atas batas OJK (0,3%/har': 'sesuai', 'Penghasilan di bawah Rp3 juta/bulan': 'sesuai', 'Indikator risiko multi-platform: > 3 apl': 'TIDAK SESUAI', 'Cicilan bulanan > 30% penghasilan (DSR)': 'TIDAK SESUAI'}
Rekomendasi: Tawaran restrukturisasi (simulasi Model 2) + kontak negosiasi langsung | Catatan: DSR melebihi batas OJK 30%
DPD aktual: 69


,tenor_bulan,bunga_harian_pct,cicilan_bulanan_idr,dsr,dsr_sesuai_ojk,pred_dpd_after,kolektibilitas_after,memenuhi_syarat
0,9,0.2,2732644,0.6326,False,69.1,Diragukan,False
1,12,0.2,2289033,0.5299,False,64.7,Diragukan,False
2,18,0.2,1845422,0.4272,False,58.7,Kurang Lancar,False
3,24,0.2,1623617,0.3758,False,50.8,Kurang Lancar,False


❌ Perpanjangan tenor tidak cukup — perlu penanganan lain (keringanan pokok/bunga atau eskalasi ke tim collection).
   Pokok maksimum OJK (DSR 30% @ tenor 24): Rp12,747,541 | pokok sekarang Rp15,970,000 | kelebihan Rp3,222,459 (20.2%) | skenario tenor 24 + pokok disesuaikan → DPD 36.3 hari (Kurang Lancar)


,Before,After (tenor 24 + pokok disesuaikan)*
Tenor (bulan),6,24
Bunga harian (%),0.23,0.2
Pokok (Rp),15970000,12747541
Cicilan bulanan (Rp),3763597,1296000
DSR,87.1%,30.0%
Prediksi DPD (hari),83.8,36.3
Kolektibilitas,Diragukan,Kurang Lancar
Status,Dalam Pengawasan,Dalam Pengawasan


Disclaimer: Simulasi menunjukkan perubahan prediksi model jika tenor atau pokok diubah. Model mempelajari pola dari data, bukan hubungan sebab-akibat, sehingga hasil simulasi bukan jaminan debitur akan membayar. Keputusan keringanan pokok tetap memerlukan persetujuan manajemen.

Cek rumus: max_principal_ojk(penghasilan, 24) = Rp12,747,541


**Hasil:** prediksi **83,8 hari → Diragukan** (aktual 69). DSR 87%, tenor sudah 6 bulan, 5 aplikasi.

**Simulasi:** di tenor 24 bulan pun DSR masih **37,6%** (> 30%), sehingga muncul **"perlu penanganan lain"**. Informasi keringanan pokok:
* `max_principal_ojk` = **Rp12.747.541** (DSR tepat 30% pada tenor 24, bunga 0,2%); pokok sekarang Rp15.970.000.
* Kelebihan pokok **Rp3.222.459 (20,2%)**.
* Skenario **tenor 24 + pokok disesuaikan** → cicilan Rp1.296.000, DSR **30,0%**, prediksi DPD **36,3 hari**. Hasilnya masih Kurang Lancar dan **tetap Dalam Pengawasan**.

**Kesimpulan kasus:** bahkan setelah tenor maksimum dan potongan pokok 20%, Model 2 masih memprediksi tunggakan > 30 hari. Debitur ini butuh penanganan di luar restrukturisasi standar, dan keputusan potongan pokok tetap perlu persetujuan manajemen. Di data test hanya **7 dari 409** debitur kelas tengah yang butuh keringanan pokok seperti ini.

## 4. Cek konsistensi Model 1 vs Model 2 pada kondisi *before*
Tabel Before memakai Model 1, sedangkan After memakai Model 2. Agar selisih Before → After benar-benar mencerminkan **perubahan tenor** dan bukan perbedaan antar model, kedua model dibandingkan pada debitur yang sama **tanpa mengubah apa pun**.

In [9]:
cmp = pd.DataFrame({'Model 1': mid['pred'], 'Model 2 (kondisi sekarang)': predict_dpd(m2, test.loc[mid.index])})
print(f"Debitur test dengan prediksi Model 1 31–90: {len(cmp)}")
display(cmp.describe().round(2).T[['mean', '50%', 'min', 'max']])
print(f"Rata-rata selisih |Model 1 − Model 2|: {np.abs(cmp['Model 1'] - cmp['Model 2 (kondisi sekarang)']).mean():.2f} hari | "
      f"rata-rata selisih bertanda (M2 − M1): {(cmp['Model 2 (kondisi sekarang)'] - cmp['Model 1']).mean():+.2f} hari")
print(f"Rata-rata penurunan DPD Before → After di simulasi (Bab 7.4 modeling): jauh lebih besar dari selisih antar model di atas.")

Debitur test dengan prediksi Model 1 31–90: 409


,mean,50%,min,max
Model 1,55.69,54.56,30.26,89.89
Model 2 (kondisi sekarang),55.39,52.67,27.84,94.46


Rata-rata selisih |Model 1 − Model 2|: 3.28 hari | rata-rata selisih bertanda (M2 − M1): -0.30 hari
Rata-rata penurunan DPD Before → After di simulasi (Bab 7.4 modeling): jauh lebih besar dari selisih antar model di atas.


**Hasil:** pada 409 debitur yang sama dan kondisi yang sama, rata-rata prediksi Model 1 **55,69** dan Model 2 **55,39 hari**. Selisih absolut rata-rata **3,28 hari**, selisih bertanda **−0,30 hari** (tidak ada bias sistematis). Penurunan Before → After di simulasi rata-rata ±31 hari (55,7 → 24,8; `modeling.ipynb` Bab 7.4), hampir 10× lebih besar dari selisih antar model. Jadi perubahan di tabel Before vs After **terutama berasal dari perubahan tenor/pokok**, bukan dari pergantian model.

## 5. Batch: skor semua debitur test (cara kerja upload CSV & task `score`)

In [10]:
out = test[['borrower_id', TARGET]].copy()
out['pred_dpd'] = test['pred'].round(1)
out['kolektibilitas'] = dpd_to_collectibility(test['pred'], THR)
out['status_kelayakan'] = dpd_to_eligibility(test['pred'], THR)
out['n_pelanggaran_ojk'] = tf[FLAG_COLS].sum(axis=1)
print(f"MAE di 2.000 debitur test (dihitung ulang dari CSV mentah): {np.abs(out[TARGET] - out['pred_dpd']).mean():.2f} hari")
display(out['status_kelayakan'].value_counts().to_frame('debitur'))
display(out.sort_values('pred_dpd', ascending=False).head(10))

MAE di 2.000 debitur test (dihitung ulang dari CSV mentah): 8.47 hari


,debitur
status_kelayakan,
Layak,1077
Tidak Layak,514
Dalam Pengawasan,409


,borrower_id,days_past_due,pred_dpd,kolektibilitas,status_kelayakan,n_pelanggaran_ojk
423,ID_DEB_DCE97092CA,180,180.0,Macet,Tidak Layak,2
146,ID_DEB_713F1A4F13,180,180.0,Macet,Tidak Layak,3
1864,ID_DEB_19FB597F1B,180,180.0,Macet,Tidak Layak,3
574,ID_DEB_DB0E8CF50F,180,180.0,Macet,Tidak Layak,2
1613,ID_DEB_AED2F312BB,180,180.0,Macet,Tidak Layak,2
999,ID_DEB_4F1D635A04,180,180.0,Macet,Tidak Layak,4
1421,ID_DEB_8E05D544C8,180,180.0,Macet,Tidak Layak,3
582,ID_DEB_E36B58760A,180,180.0,Macet,Tidak Layak,3
994,ID_DEB_4051EAE1D3,180,180.0,Macet,Tidak Layak,2
163,ID_DEB_D19F6C979B,180,180.0,Macet,Tidak Layak,3


## Kesimpulan inference
1. Kedua model dimuat dari `.pkl` **tanpa training ulang**, dan versi scikit-learn dicek cocok dengan metadata. MAE yang dihitung ulang dari CSV mentah (**8,47 hari**) sama dengan hasil `modeling.ipynb`.
2. Input cukup **11 kolom mentah**. Fitur, flag OJK, pemetaan status (batas Lancar 8 hari dari CV), dan simulasi memakai fungsi `features.py` yang sama dengan aplikasi dan DAG.
3. Kelima kasus menghasilkan hal berbeda:
   * Kasus 1: Lancar.
   * Kasus 2: opsi tenor pendek (6 bulan).
   * Kasus 3: tenor 9 bulan + bunga turun ke batas OJK.
   * Kasus 4: peringatan ekstrapolasi.
   * Kasus 5: "perlu penanganan lain" + pokok maksimum OJK.
4. Distribusi status di 2.000 debitur test: **Layak 1.077 · Dalam Pengawasan 409 · Tidak Layak 514**.